Mooncake moves model key-value cache bytes between machines (the transfer engine, `mooncake.engine`) and pools memory across machines into one cache (the store, `mooncake.store`, with its `mooncake_master` program). The package installs as `mooncake`. Everything below runs on ONE machine over TCP with no metadata server; Example 5 needs a GPU. What cannot be shown on one machine: the RDMA transport and any transfer between two machines.

## Example 1 — a transfer engine starts over TCP with no metadata server and allocates a buffer

In [ ]:
# Adapted from Mooncake (Apache-2.0) v0.3.13.post1, mooncake-wheel/tests/transfer_engine_target.py (initialize, then a buffer).
from mooncake.engine import TransferEngine
target = TransferEngine()
# "P2PHANDSHAKE": the two ends exchange their details directly, so no metadata server has to run.
rc = target.initialize("127.0.0.1:12345", "P2PHANDSHAKE", "tcp", "")
assert rc == 0, f"initialize returned {rc}"
port = target.get_rpc_port()
assert 0 < port < 65536, port
size = 1 << 20
buffer_address = target.allocate_managed_buffer(size)
assert buffer_address != 0, "the engine could not allocate a buffer"
assert target.free_managed_buffer(buffer_address, size) == 0
print(f"OK -- transfer engine up on 127.0.0.1:{port} (TCP, peer-to-peer handshake)")

## Example 2 — known bytes written from one engine to another over TCP, checked on the target and read back

In [ ]:
# Adapted from Mooncake (Apache-2.0) v0.3.13.post1, mooncake-wheel/tests/transfer_engine_initiator_test.py (write, wipe, read back).
import hashlib
from mooncake.engine import TransferEngine
size = 1 << 20
ends = []
for port in (12351, 12352):
    te = TransferEngine()
    assert te.initialize(f"127.0.0.1:{port}", "P2PHANDSHAKE", "tcp", "") == 0
    ends.append((te, f"127.0.0.1:{te.get_rpc_port()}", te.allocate_managed_buffer(size)))
(a, _, source), (b, b_name, destination) = ends
payload = hashlib.sha256(b"mooncake smoke test").digest() * (size // 32)
assert a.write_bytes_to_buffer(source, payload, size) == 0
assert b.write_bytes_to_buffer(destination, bytes(size), size) == 0  # control: the target starts as zeros
assert a.transfer_sync_write(b_name, source, destination, size) == 0
assert b.read_bytes_from_buffer(destination, size) == payload, "the bytes did not arrive in the target's buffer"
assert a.write_bytes_to_buffer(source, bytes(size), size) == 0  # wipe the local copy before reading back
assert a.transfer_sync_read(b_name, source, destination, size) == 0
assert a.read_bytes_from_buffer(source, size) == payload, "the bytes read back differ from the bytes sent"
print(f"OK -- {size} bytes written to {b_name} over TCP, seen in the target's buffer, read back and equal")

## Example 3 — the store's replication settings: values go in and come back, and bad values are refused

In [ ]:
# Adapted from Mooncake (Apache-2.0) v0.3.13.post1, mooncake-wheel/tests/test_distributed_object_store.py (TestReplicateConfig).
from mooncake.store import ReplicateConfig
config = ReplicateConfig()
assert config.replica_num == 1 and config.soft_pin_ttl_ms is None
config.replica_num = 2
config.soft_pin_ttl_ms = 1000
assert (config.replica_num, config.soft_pin_ttl_ms) == (2, 1000)
config.soft_pin_ttl_ms = None
assert config.soft_pin_ttl_ms is None
for bad in (-1, 1 << 64):
    try:
        config.soft_pin_ttl_ms = bad
    except TypeError:
        continue
    raise AssertionError(f"soft_pin_ttl_ms accepted {bad}")
print("OK -- ReplicateConfig keeps replica_num and soft_pin_ttl_ms, and refuses a negative or oversized value")

## Example 4 — one put and one get through the store, with the package's own `mooncake_master` started locally

In [ ]:
# Adapted from Mooncake (Apache-2.0) v0.3.13.post1, mooncake-wheel/tests/test_distributed_object_store.py (get_client, put, get).
import os, shutil, subprocess, time
from mooncake.store import MooncakeDistributedStore
assert shutil.which("mooncake_master"), "the package's mooncake_master command is not on PATH"
master = subprocess.Popen(["mooncake_master", "--port=50619", "--metrics_port=9619", "--enable_metric_reporting=false"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
try:
    time.sleep(3)
    assert master.poll() is None, f"mooncake_master exited with {master.returncode}"
    store = MooncakeDistributedStore()
    assert store.setup("127.0.0.1", "P2PHANDSHAKE", 64 << 20, 32 << 20, "tcp", "", "127.0.0.1:50619") == 0
    key, value = f"smoke_test_{os.getpid()}", os.urandom(1 << 20)
    assert store.is_exist(key) == 0  # control: nothing is there before the put
    assert store.put(key, value) == 0
    assert store.get(key) == value, "the store returned different bytes"
    assert len(store.get(key + "_never_written")) == 0
    store.close()
finally:
    master.terminate()
    master.wait(timeout=30)
print("OK -- 1 MiB put into the store and read back equal; a key never written returns nothing")

## Example 5 — GPU memory registered with the engine and moved through it (needs a GPU)

In [ ]:
# Adapted from Mooncake (Apache-2.0) v0.3.13.post1, mooncake-wheel/tests/test_transfer_on_cuda.py (register_memory on tensors);
# TCP instead of that test's RDMA, so it runs on one machine with no RDMA network card.
import torch
from mooncake.engine import TransferEngine
assert torch.cuda.is_available(), "requires-gpu cell ran but torch sees no CUDA device"
te = TransferEngine()
assert te.initialize("127.0.0.1:12361", "P2PHANDSHAKE", "tcp", "") == 0
own_name = f"127.0.0.1:{te.get_rpc_port()}"
count = 4 << 20
source = (torch.arange(count, device="cuda") % 251).to(torch.uint8)
destination = torch.zeros(count, dtype=torch.uint8, device="cuda")
for tensor in (source, destination):
    assert te.register_memory(tensor.data_ptr(), tensor.nbytes) == 0
assert not torch.equal(source, destination)  # control: they differ before the transfer
assert te.transfer_sync_write(own_name, source.data_ptr(), destination.data_ptr(), count) == 0
torch.cuda.synchronize()
assert torch.equal(source, destination), "the GPU bytes did not arrive"
for tensor in (source, destination):
    te.unregister_memory(tensor.data_ptr())
print(f"OK -- {count} bytes of GPU memory on {torch.cuda.get_device_name(0)} moved through the engine and equal")